In [10]:
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from typing import TypedDict
import time
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [11]:
class BlogState(TypedDict):
    title : str
    outline  : str
    content : str

In [12]:
def generate_outline(state: BlogState ) -> BlogState :
    title = state["title"]
    prompt = f"You are an expert blog writer. When provided with the title generate outline for the topic. Generate outlines for the blog topic {title}"
    outline = model.invoke(prompt).content
    print("Outline generated" , outline)
    state["outline"] = outline
    time.sleep(2)
    return state

def generate_blog(state : BlogState) -> BlogState:
    title = state["title"]
    outline = state["outline"]
    prompt = f"""You are a professional blog writer.

Given the blog title and a structured outline, generate a well-written, SEO-friendly blog article. Follow the tone of a helpful and clear human writer. The blog should be informative, engaging, and easy to read.
## Blog Title:
{title}
## Blog Outline:
{outline}
## Guidelines:
- Begin with a compelling introduction (2–3 short paragraphs).
- For each heading in the outline, generate 2–4 paragraphs of content.
- Use examples, explanations, and transitions where relevant.
- End with a short conclusion or call to action.
- Use markdown formatting with clear **headings (##)**.
- Keep paragraphs short (3–5 lines) and readable.
- Include bullet points or numbered lists if the section calls for it.
- Do not hallucinate data or statistics; if unsure, mention "For example…" without inventing numbers.
Now, write the full blog."""
    content = model.invoke(prompt).content
    print("Blog Generated" , content)
    state["content"] = content
    time.sleep(2)
    return state


In [13]:
graph = StateGraph(BlogState)
graph.add_node( "generate_outline", generate_outline)
graph.add_node( "generate_blog", generate_blog)
graph.add_edge(START , "generate_outline")
graph.add_edge("generate_outline", "generate_blog")
graph.add_edge("generate_blog", END)

workflow = graph.compile()


In [14]:
initial_state = {
    "title" : "Artificial Intelligence will replace Software Engineer"
}
res = workflow.invoke(initial_state)
print("contents of the blog", res["content"] )

Outline generated Okay, here's an outline for a blog post titled "Artificial Intelligence Will Replace Software Engineers":

**I. Introduction**

*   **Hook:** Start with a captivating question or a bold statement to grab the reader's attention (e.g., "Are software engineers about to become obsolete?").
*   **Briefly introduce the topic:** Acknowledge the growing concern about AI's impact on various professions, specifically focusing on software engineering.
*   **State the blog post's purpose/thesis:** This could be a statement arguing for or against the replacement of software engineers by AI, or a more nuanced position (e.g., "While AI will significantly transform the role of software engineers, complete replacement is unlikely in the foreseeable future.").
*   **Outline the main points to be discussed:** Briefly mention the key arguments that will be explored in the blog post.

**II. Understanding the Current Capabilities of AI in Software Development**

*   **AI-Powered Code Gener